# Comandos para realização do trabalho da matéria de Big Data com uso da biblioteca PySpark.

Este notebook foi projetado para guiar os alunos na realização das práticas de Big Data utilizando PySpark. Certifique-se de seguir cada etapa cuidadosamente para garantir a correta execução das atividades.

Seu trabalho começará na célula 5. Execute as 4 primeiras células para iniciar a atividade.

## <font color=red>Observação importante:</font>

<font color=yellow>Trabalho realizado com uso da biblioteca pandas não será aceito!</font>

## Upload do arquivo `imdb-reviews-pt-br.csv` para dentro do Google Colab

Aqui, você fará o download do dataset necessário para as atividades. Certifique-se de que o arquivo foi descompactado corretamente antes de prosseguir.

In [2]:
!wget https://raw.githubusercontent.com/N-CPUninter/Big_Data/main/data/imdb-reviews-pt-br.zip -O imdb-reviews-pt-br.zip
!unzip imdb-reviews-pt-br.zip
!rm imdb-reviews-pt-br.zip

--2026-06-26 19:50:15--  https://raw.githubusercontent.com/N-CPUninter/Big_Data/main/data/imdb-reviews-pt-br.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49549692 (47M) [application/zip]
Saving to: ‘imdb-reviews-pt-br.zip’

imdb-reviews-pt-br. 100%[===================>]  47.25M  --.-KB/s    in 0.1s    

2026-06-26 19:50:19 (413 MB/s) - ‘imdb-reviews-pt-br.zip’ saved [49549692/49549692]

Archive:  imdb-reviews-pt-br.zip
  inflating: imdb-reviews-pt-br.csv  


## Instalação manual das dependências para uso do pyspark no Google Colab

Esta etapa garante que todas as bibliotecas necessárias para o PySpark sejam instaladas no Google Colab.

In [3]:
!apt-get install -y openjdk-17-jdk-headless
!pip install pyspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk-headless is already the newest version (17.0.19+10-1~22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


## Importar, instanciar e criar a SparkSession

A SparkSession é o ponto de entrada para usar o PySpark. Certifique-se de configurar corretamente o nome do aplicativo e o master.

In [4]:
import setuptools
import os
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")


appName = "PySpark Trabalho de Big Data"
master = "local"

spark = SparkSession.builder.appName(appName).master(master).getOrCreate()

## Criar spark dataframe do CSV utilizando o método read.csv do spark

Não altere este código e use o dataframe imdb_df criado aqui em todo o seu trabalho. A criação de um dataframe diferente deste poderá causar erros na coluna sentiment e isso refletirá em erros de resposta das questões.

In [5]:
imdb_df = spark.read.csv('imdb-reviews-pt-br.csv',
                         header=True,
                         quote="\"",
                         escape="\"",
                         encoding="UTF-8")

# Questão 1

Nesta questão, você irá calcular a soma dos IDs para entradas onde o sentimento ('sentiment') é 'neg'.

### Objetivo:
- Usar a coluna 'sentiment' como chave e somar os valores da coluna 'id'.

## Criar funções de MAP:
- Criar função para mapear o "sentiment" como chave e o "id" como valor do tipo inteiro

A função map irá transformar cada linha do dataframe em uma **tupla** (chave-valor), onde:
- Chave: coluna 'sentiment'
- Valor: coluna 'id' convertida para inteiro.

In [10]:
def map1(x):

    return (x['sentiment'], int(x['id']))

In [7]:
imdb_df.first()

Row(id='1', text_en='Once again Mr. Costner has dragged out a movie for far longer than necessary. Aside from the terrific sea rescue sequences, of which there are very few I just did not care about any of the characters. Most of us have ghosts in the closet, and Costners character are realized early on, and then forgotten until much later, by which time I did not care. The character we should really care about is a very cocky, overconfident Ashton Kutcher. The problem is he comes off as kid who thinks hes better than anyone else around him and shows no signs of a cluttered closet. His only obstacle appears to be winning over Costner. Finally when we are well past the half way point of this stinker, Costner tells us all about Kutchers ghosts. We are told why Kutcher is driven to be the best with no prior inkling or foreshadowing. No magic here, it was all I could do to keep from turning it off an hour in.', text_pt='Mais uma vez, o Sr. Costner arrumou um filme por muito mais tempo do q

## Cria funções de REDUCE:

- Criar função de reduce para somar os IDs por "sentiment".

A função reduce irá somar os valores dos IDs agrupados por chave ('sentiment').

In [11]:
def reduceByKey1(x,y):

    return x + y

In [13]:
resultado_q1 = imdb_df.rdd \
    .map(map1) \
    .reduceByKey(reduceByKey1) \
    .collect()

print("======================================")
print("RU: 2071642")
print("QUESTÃO 1 - SOMATÓRIO DOS IDs")
print("======================================")
print(resultado_q1)

for sentimento, soma in resultado_q1:
    if sentimento == "neg":
        print("Soma dos IDs negativos:", soma)

print("======================================")

RU: 2071642
QUESTÃO 1 - SOMATÓRIO DOS IDs
[('neg', 459568555), ('pos', 763600041)]
Soma dos IDs negativos: 459568555


## Aplicação do map/reduce e visualização do resultado

Aqui, você aplicará as funções de map e reduce ao dataframe Spark para calcular os resultados. Não se esqueça de usar o método `.collect()` para visualizar os resultados.

# Questão 2:

Nesta questão, você irá calcular a diferença no número total de palavras entre textos negativos em português e inglês.

### Objetivo:
- Contar as palavras em cada idioma (colunas 'text_pt' e 'text_en') para entradas onde o sentimento ('sentiment') é 'neg'.
- Subtrair o total de palavras em inglês do total em português.

## Criar funções de MAP:
- Criar função para mapear o "sentiment" como chave de uma tupla principal e como valor uma outra tupla com a soma das palavras de cada idioma como valor.

A função map irá transformar cada linha do dataframe em uma tupla (chave-valor), onde:
- Chave: coluna 'sentiment'
- Valor: Nova tupla com:
  - Elemento 0: soma das palavras da coluna 'text_en'
  - Elemento 1: soma das palavras da coluna 'text_pt'

OU
- Chave: coluna 'sentiment'
- Valor: (soma das palavras da coluna 'text_pt') - (soma das palavras da coluna 'text_en')
  

Para contar as palavras deve-se primeiro separar os textos em uma lista de palavras para então descobrir o tamanho desta lista.
Dicas:

1. Use o método .split() e não .split(" ") de string para separar as palavras em uma lista ou use a função split(coluna de texto, regex) do pyspark com o regex igual à "[ ]+" ou "\s+"
2. Use len() para descobrir o tamanho da lista de palavras.

In [14]:
def map2(x):
    sentiment = x['sentiment']
    qtd_en = len(x['text_en'].split())
    qtd_pt = len(x['text_pt'].split())

    return (sentiment, (qtd_en, qtd_pt))

## Cria funções de REDUCE:

- Criar função de reduce para somar o numero de palavras de cada texto português e inglês por "sentiment" (dependerá de como você optou por fazer sua função map2).

A função reduce irá somar os valores das quantidades de palavras agrupados por chave ('sentiment').

In [16]:
def reduceByKey2(x,y):
   return (x[0] + y[0], x[1] + y[1])

## Aplicação do map/reduce e visualização do resultado

1. Aplicar o map/reduce no seu dataframe spark e realizar o collect() ao final
2. Selecionar os dados referentes aos textos negativos para realizar a subtração.
3. Realizar a subtração das contagens de palavras dos textos negativos para obter o resultado final

In [29]:
resultado = (
    imdb_df
    .rdd
    .filter(lambda x: x['sentiment'] == 'neg')
    .map(map2)
    .reduceByKey(reduceByKey2)
    .collect()
)

print("RU: 2071642")
print(resultado)

total_en = resultado[0][1][0]
total_pt = resultado[0][1][1]

diferenca = total_pt - total_en

print("Total de palavras em inglês:", total_en)
print("Total de palavras em português:", total_pt)
print("Diferença (PT - EN):", diferenca)

RU: 2071642
[('neg', (5400324, 5455273))]
Total de palavras em inglês: 5400324
Total de palavras em português: 5455273
Diferença (PT - EN): 54949


In [27]:
# Linha de código para aplicar o map/reduce no seu dataframe spark
resultado = imdb_df.rdd.map(map2).reduceByKey(reduceByKey2).collect()
# Coloque aqui suas linhas de código final para imprimir o resultado.
# Não esqueça seu RU:

# Filtrar apenas sentimentos negativos
resultado = (
    imdb_df
    .rdd
    .filter(lambda x: x['sentiment'] == 'neg')
    .map(map2)
    .reduceByKey(reduceByKey2)
    .collect()
)

# imprimir resultado


# coloque seu RU aqui
print("======================================")
print("RU: 2071642")
print("QUESTÃO 2 - CONTAGEM DE PALAVRAS")
print("======================================")
print(resultado)


RU: 2071642
QUESTÃO 2 - CONTAGEM DE PALAVRAS
[('neg', (5400324, 5455273))]


___

##### Esses codigos serão excluidos pois são repetidos

In [30]:
total_en = resultado[0][1][0]
total_pt = resultado[0][1][1]

diferenca = total_pt - total_en

print("Total de palavras em inglês:", total_en)
print("Total de palavras em português:", total_pt)
print("Diferença (PT - EN):", diferenca)

Total de palavras em inglês: 5400324
Total de palavras em português: 5455273
Diferença (PT - EN): 54949


In [22]:
imdb_df.filter("lower(trim(sentiment)) = 'neg'")


DataFrame[id: string, text_en: string, text_pt: string, sentiment: string]

In [21]:
from pyspark.sql.functions import col, lower, trim

imdb_df.filter(lower(trim(col("sentiment"))) == "neg")


DataFrame[id: string, text_en: string, text_pt: string, sentiment: string]

In [19]:
imdb_df.select("sentiment").distinct().show()

+---------+
|sentiment|
+---------+
|      pos|
|      neg|
+---------+



---